# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [9]:
ПРІЗВИЩЕ_ІМЯ = 'Соломія Лящук'      
ГРУПА        = 'ШІ-33'
НОМЕР        = 6      

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [10]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 6   ·   Соломія Лящук   ·   ШІ-33

Ваші вісім стовпців:
   1. ПІП
   2. Примітка
   3. Перебіг
   4. Дата розтину
   5. Ліжко днів
   6. Т вище38
   7. МНВ1
   8. Креат2

Ваша пара для аналізу зв’язку: Ліжко днів та Вік

Ваше додаткове питання:
   Скільки стовпців файлу є ідентифікаторами? Як їх розпізнати автоматично, не дивлячись на назви?


In [11]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,ПІП,Примітка,Перебіг,Дата розтину,Ліжко днів,Т вище38,МНВ1,Креат2
0,Федорюк Петро Юрійович,NaN,важкий,2022-03-11,14,4 дні до госп,3.10,92
1,Загул Григорій Сергійович,"отримував актемра, часті носові кровотечі, втрата нюху",важкий,NaT,22,5 до гос,1.22,73
2,Бондара Лариса Павлівна,часта втрата свідомосі в стаціонарі,важкий,NaT,20,10 до гос,1.03,117.2
3,Ватаманюка Уляна Дмитрівна,панкреонекроз,важкий,NaT,20,9 до гос,1.69,105.1
4,Бабійа Лариса Олегівна,NaN,важкий,NaT,27,NaN,1.15,101
5,Ковалика ганна Андріївна,NaN,важкий,2021-12-26,9(5 у реан),NaN,0.90,172
6,Височан Віктор Володимирович,"глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",важкий,NaT,18,7 дні до госп,1.09,97.8
7,Дзюбаа Ірина Іванівна,NaN,серед важ,NaT,10,9 днів до госп,0.94,107


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [12]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   ПІП           121 non-null    object        
 1   Примітка      45 non-null     object        
 2   Перебіг       121 non-null    object        
 3   Дата розтину  61 non-null     datetime64[ns]
 4   Ліжко днів    120 non-null    object        
 5   Т вище38      77 non-null     object        
 6   МНВ1          90 non-null     float64       
 7   Креат2        102 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 7.7+ KB


In [13]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- ПІП
    непорожніх 121 з 121 | унікальних 121 | типи: str×121
    приклади: ["'Федорюк Петро Юрійович'", "'Загул Григорій Сергійович'", "'Бондара Лариса Павлівна'", "'Ватаманюка Уляна Дмитрівна'"]

--- Примітка
    непорожніх 45 з 121 | унікальних 44 | типи: str×45
    приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]

--- Перебіг
    непорожніх 121 з 121 | унікальних 3 | типи: str×121
    приклади: ["'важкий'", "'серед важ'", "'вкрай важкий '"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- Ліжко днів
    непорожніх 120 з 121 | унікальних 40 | типи: int×119, str×1
    приклади: ['14', '22', '20', '27']

--- Т вище38
    непорожніх 77 з 121 | унікаль

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [14]:
ПАСПОРТ = {
    'ПІП': ('ідентифікатор', 'Це унікальне значення, що вказує на конкретного пацієнта, а не є величиною для математичних операцій.'),
    'Примітка': ('номінальна', 'Містить якісні описи станів чи лікування, які можна згрупувати, але неможливо математично впорядкувати.'),
    'Перебіг': ('порядкова', 'Стани («важкий», «середній») мають чіткий природний порядок, але числова відстань між ними невідома.'),
    'Дата розтину': ('інтервальна', 'Для дат немає абсолютного нуля, але операція віднімання (різниця у днях між датами) має точний зміст.'),
    'Ліжко днів': ('абсолютна', 'Підрахунок кількості днів має природний нуль та природну одиницю лічби, тому їх можна порівнювати у разах.'),
    'Т вище38': ('абсолютна', 'За своїм фізичним змістом це лічильник кількості днів із лихоманкою, що має природний нуль та одиницю лічби.'),
    'МНВ1': ('відношень', 'Це лабораторний показник, який має справжній нуль, і для нього повноцінно мають сенс відношення (ділення).'),
    'Креат2': ('відношень', 'Концентрація речовини в крові, що вимірюється на шкалі з абсолютним нулем і допускає будь-які арифметичні дії.')
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,ПІП,ідентифікатор,"Це унікальне значення, що вказує на конкретного пацієнта..."
1,Примітка,номінальна,"Містить якісні описи станів чи лікування, які можна згру..."
2,Перебіг,порядкова,"Стани («важкий», «середній») мають чіткий природний поря..."
3,Дата розтину,інтервальна,"Для дат немає абсолютного нуля, але операція віднімання ..."
4,Ліжко днів,абсолютна,Підрахунок кількості днів має природний нуль та природну...
5,Т вище38,абсолютна,За своїм фізичним змістом це лічильник кількості днів із...
6,МНВ1,відношень,"Це лабораторний показник, який має справжній нуль, і для..."
7,Креат2,відношень,"Концентрація речовини в крові, що вимірюється на шкалі з..."


In [15]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [16]:
import numpy as np
import re

def clean_series(s):
    """Допоміжна функція для витягнення чисел із текстових полів (наприклад, '4 дні до госп')"""
    def parse_val(val):
        if pd.isna(val):
            return np.nan
        if isinstance(val, (int, float)):
            return float(val)
        m = re.search(r'(\d+(?:[\.,]\d+)?)', str(val))
        if m:
            return float(m.group(1).replace(',', '.'))
        return np.nan
    return s.apply(parse_val)


def допустима_статистика(стовпець, шкала):
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р

    if len(з) > 0:
        р['мода'] = з.mode()[0] if not з.mode().empty else None

    if ш in ('порядкова', 'інтервальна', 'відношень', 'абсолютна'):
        if ш == 'порядкова':
            order_map = {'легкий': 1, 'середній': 2, 'серед важ': 2, 'важкий': 3, 'вкрай важкий': 4}
            inv_map = {1: 'легкий', 2: 'середній', 3: 'важкий', 4: 'вкрай важкий'}
            ranks = з.astype(str).str.strip().map(order_map).dropna()
            if not ranks.empty:
                med_r = int(round(ranks.median()))
                р['медіана'] = inv_map.get(med_r, 'важкий')
        else:
            if not pd.api.types.is_datetime64_any_dtype(з):
                num_з = clean_series(з).dropna()
                р['медіана'] = round(float(num_з.median()), 2)
                р['25% квантиль'] = round(float(num_з.quantile(0.25)), 2)
                р['75% квантиль'] = round(float(num_з.quantile(0.75)), 2)

    if ш in ('інтервальна', 'відношень', 'абсолютна'):
        if pd.api.types.is_datetime64_any_dtype(df[стовпець]):
            dt_vals = df[стовпець].dropna()
            mean_dt = dt_vals.mean()
            std_days = (dt_vals - mean_dt).dt.total_seconds().std() / 86400.0
            р['середнє'] = str(mean_dt.date())
            р['ст. відхилення (днів)'] = round(std_days, 2)
        else:
            num_з = clean_series(df[стовпець]).dropna()
            р['середнє'] = round(float(num_з.mean()), 2)
            р['ст. відхилення'] = round(float(num_з.std()), 2)

    if ш in ('відношень', 'абсолютна'):
        if not pd.api.types.is_datetime64_any_dtype(df[стовпець]):
            num_з = clean_series(df[стовпець]).dropna()
            m = num_з.mean()
            if m != 0:
                р['коеф. варіації (%)'] = round(float((num_з.std() / m) * 100), 2)

    return р

for c in МОЇ:
    print(f'--- {c} ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- ПІП (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 121}

--- Примітка (номінальна)
    {'непорожніх': 45, 'унікальних': 44, 'мода': 'Поліалергетик'}

--- Перебіг (порядкова)
    {'непорожніх': 121, 'унікальних': 3, 'мода': 'важкий', 'медіана': 'важкий'}

--- Дата розтину (інтервальна)
    {'непорожніх': 61, 'унікальних': 50, 'мода': Timestamp('2022-06-07 00:00:00'), 'середнє': '2022-04-06', 'ст. відхилення (днів)': np.float64(72.57)}

--- Ліжко днів (абсолютна)
    {'непорожніх': 120, 'унікальних': 40, 'мода': 10, 'медіана': 15.0, '25% квантиль': 10.0, '75% квантиль': 22.0, 'середнє': 17.88, 'ст. відхилення': 12.94, 'коеф. варіації (%)': 72.34}

--- Т вище38 (абсолютна)
    {'непорожніх': 77, 'унікальних': 28, 'мода': 3, 'медіана': 4.0, '25% квантиль': 3.0, '75% квантиль': 5.0, 'середнє': 4.22, 'ст. відхилення': 1.85, 'коеф. варіації (%)': 43.93}

--- МНВ1 (відношень)
    {'непорожніх': 90, 'унікальних': 53, 'мода': np.float64(1.03), 'медіана': 1.19, '25% квантиль': 1.05, '7

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [17]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ПІП,121,121,Федорюк Петро Юрійович,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Примітка,45,44,Поліалергетик,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Перебіг,121,3,важкий,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049088,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
Ліжко днів,120.0,40.0,10.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Т вище38,77.0,28.0,3.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
МНВ1,90.0,NaN,NaN,NaN,1.333778,0.8,1.0525,1.19,1.34,10.6,1.032131
Креат2,102.0,72.0,77.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
pd.set_option('display.max_colwidth', None)
mean_course = round(float(df['Перебіг'].str.strip().map({'серед важ': 1, 'важкий': 2, 'вкрай важкий': 3}).mean()), 2)

dt_series = df['Дата розтину'].dropna().astype('int64')
cv_date = round(float((dt_series.std() / dt_series.mean()) * 100), 4)

mean_note_len = round(float(df['Примітка'].dropna().str.len().mean()), 2)

БЕЗГЛУЗДІ = [
    ('Перебіг', 'Середнє арифметичне', mean_course, 'Порядкова шкала не має кількісного вираження; ''обчислення середнього для категорій є некоректним.'),
    ('Дата розтину', 'Коефіцієнт варіації (%)', cv_date, 'Інтервальна шкала не має абсолютного нуля, тому відношення стандартного відхилення до середнього позбавлене змісту.'),
    ('Примітка', 'Середня довжина тексту', mean_note_len, 'Номінальна шкала описує якісний стан; середня кількість символів описує стиль запису, а не медичний показник.')
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Перебіг,Середнє арифметичне,1.7600,Порядкова шкала не має кількісного вираження; обчислення середнього для категорій є некоректним.
1,Дата розтину,Коефіцієнт варіації (%),0.3802,"Інтервальна шкала не має абсолютного нуля, тому відношення стандартного відхилення до середнього позбавлене змісту."
2,Примітка,Середня довжина тексту,74.4400,"Номінальна шкала описує якісний стан; середня кількість символів описує стиль запису, а не медичний показник."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [23]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Ліжко днів × Вік

Ліжко днів: шкала — абсолютна
Вік: шкала — не у вашому списку

    Ліжко днів  Вік
0           14   55
1           22   43
2           20   54
3           20   68
4           27   65
5  9(5 у реан)   66


In [ ]:
ОБРАНА_МІРА = 'кореляція Пірсона'
ЧОМУ_САМЕ_ВОНА = 'Обидві змінні (Ліжко днів та Вік) належать до кількісної (абсолютної) шкали з природним нулем. Згідно з матрицею шкал, для пари відношень × відношень обирається кореляція Пірсона.'

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

import scipy.stats as stats
import re

def clean_num(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    m = re.search(r'(\d+(?:[\.,]\d+)?)', str(val))
    return float(m.group(1).replace(',', '.')) if m else np.nan

df_pair = df[[A, B]].copy()
df_pair[A] = df_pair[A].apply(clean_num)
df_pair[B] = df_pair[B].apply(clean_num)
data = df_pair.dropna()

r, p = stats.pearsonr(data[A], data[B])
print(f"Коефіцієнт кореляції Пірсона між '{A}' та '{B}': r = {r:.4f} (p-value = {p:.4f})")


Коефіцієнт кореляції Пірсона між 'Ліжко днів' та 'Вік': r = -0.0996 (p-value = 0.2791)


In [26]:
course_map = {'серед важ': 1, 'важкий': 2, 'вкрай важкий': 3}
df_bad = df[['Перебіг', 'Вік']].dropna().copy()
df_bad['Перебіг_число'] = df_bad['Перебіг'].str.strip().map(course_map)
df_bad['Вік_число'] = df_bad['Вік'].apply(clean_num)
df_bad = df_bad.dropna()

r_bad, p_bad = stats.pearsonr(df_bad['Перебіг_число'], df_bad['Вік_число'])

print(f"Формально розрахований Пірсон для 'Перебіг' (порядкова) × 'Вік': r = {r_bad:.4f}\n")
print("ЧОМУ ЦЕ ЧИСЛО НЕ МОЖНА ВИКОРИСТОВУВАТИ:")
print("1. Змінна 'Перебіг' належить до порядкової шкали. Інтервали між рангами (1, 2, 3) не є однаковими.")
print("2. Кореляція Пірсона вимагає метричних шкал і розраховує лінійний зв'язок на основі справжніх числових відстаней.")
print("3. Кодування порядкових категорій числами є умовним, тому отриманий коефіцієнт є викривленим і позбавленим математичного сенсу.")

Формально розрахований Пірсон для 'Перебіг' (порядкова) × 'Вік': r = 0.3541

ЧОМУ ЦЕ ЧИСЛО НЕ МОЖНА ВИКОРИСТОВУВАТИ:
1. Змінна 'Перебіг' належить до порядкової шкали. Інтервали між рангами (1, 2, 3) не є однаковими.
2. Кореляція Пірсона вимагає метричних шкал і розраховує лінійний зв'язок на основі справжніх числових відстаней.
3. Кодування порядкових категорій числами є умовним, тому отриманий коефіцієнт є викривленим і позбавленим математичного сенсу.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [27]:
ПІДГОТОВКА = {
    'ПІП': 'Ідентифікатор: вилучаємо з моделі, оскільки персональне ім’я не має узагальнюючої здатності.',
    'Примітка': 'Номінальний нотаток: вилучаємо з моделі або кодуємо через TF-IDF для аналізу тексту.',
    'Перебіг': 'Порядкова шкала: застосовуємо Ordinal Encoding з порядком (серед важ: 1, важкий: 2, вкрай важкий: 3).',
    'Дата розтину': 'Інтервальна шкала: конвертуємо у кількість днів від референтної дати або вилучаємо.',
    'Ліжко днів': 'Шкала відношень: виконуємо MinMax-нормалізацію (MinMaxScaler) після чистки некоректних символів.',
    'Т вище38': 'Шкала відношень: масштабуємо за допомогою StandardScaler або MinMaxScaler.',
    'МНВ1': 'Шкала відношень: застосовуємо MinMaxScaler або Z-score стандартизацію.',
    'Креат2': 'Шкала відношень: проводимо MinMax-нормалізацію або Z-score стандартизацію.'
}

for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,ПІП,ідентифікатор,"Ідентифікатор: вилучаємо з моделі, оскільки персональне ім’я не має узагальнюючої здатності."
1,Примітка,номінальна,Номінальний нотаток: вилучаємо з моделі або кодуємо через TF-IDF для аналізу тексту.
2,Перебіг,порядкова,"Порядкова шкала: застосовуємо Ordinal Encoding з порядком (серед важ: 1, важкий: 2, вкрай важкий: 3)."
3,Дата розтину,інтервальна,Інтервальна шкала: конвертуємо у кількість днів від референтної дати або вилучаємо.
4,Ліжко днів,абсолютна,Шкала відношень: виконуємо MinMax-нормалізацію (MinMaxScaler) після чистки некоректних символів.
5,Т вище38,абсолютна,Шкала відношень: масштабуємо за допомогою StandardScaler або MinMaxScaler.
6,МНВ1,відношень,Шкала відношень: застосовуємо MinMaxScaler або Z-score стандартизацію.
7,Креат2,відношень,Шкала відношень: проводимо MinMax-нормалізацію або Z-score стандартизацію.


In [28]:
import re

course_map = {'серед важ': 1, 'важкий': 2, 'вкрай важкий': 3}
df['Перебіг_encoded'] = df['Перебіг'].str.strip().map(course_map)

def clean_num(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    m = re.search(r'(\d+(?:[\.,]\d+)?)', str(val))
    return float(m.group(1).replace(',', '.')) if m else np.nan

beds_clean = df['Ліжко днів'].apply(clean_num)
df['Ліжко_днів_normalized'] = (beds_clean - beds_clean.min()) / (beds_clean.max() - beds_clean.min())

df_for_model = df[МОЇ].drop(columns=['ПІП'])

print("1. Порядкова шкала ('Перебіг') -> Ordinal Encoding:")
print(df[['Перебіг', 'Перебіг_encoded']].head(4))

print("\n2. Шкала відношень ('Ліжко днів') -> MinMax Normalization:")
print(df[['Ліжко днів', 'Ліжко_днів_normalized']].head(4))

print("\n3. Ідентифікатор ('ПІП') -> вилучено з навчальної вибірки.")
print("Залишені стовпці у df_for_model:", list(df_for_model.columns))


1. Порядкова шкала ('Перебіг') -> Ordinal Encoding:
  Перебіг  Перебіг_encoded
0  важкий                2
1  важкий                2
2  важкий                2
3  важкий                2

2. Шкала відношень ('Ліжко днів') -> MinMax Normalization:
  Ліжко днів  Ліжко_днів_normalized
0         14               0.166667
1         22               0.269231
2         20               0.243590
3         20               0.243590

3. Ідентифікатор ('ПІП') -> вилучено з навчальної вибірки.
Залишені стовпці у df_for_model: ['Примітка', 'Перебіг', 'Дата розтину', 'Ліжко днів', 'Т вище38', 'МНВ1', 'Креат2']


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [29]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [30]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [31]:
ВИПРАВЛЕННЯ = [
    (
        "Розділ 1. Описова статистика",
        "Обчислення середнього арифметичного для порядкової шкали (Перебіг / Ступінь КТ)",
        "Категорії порядкової шкали не мають однакових інтервалів між собою, тому середнє є арифметично безглуздим.",
        "Застосувати медіану або моду з аналізом частотного розподілу категорій."
    ),
    (
        "Розділ 2. Описова статистика",
        "Обчислення середнього та стандартного відхилення для номінальної шкали (Стать / помер/жив)",
        "Номінальні коди є лише якісними мітками, тому додавання та ділення над ними не мають математичного сенсу.",
        "Використовувати частоти, відсоткові частки (пропорції) та моду."
    ),
    (
        "Розділ 3. Варіативність",
        "Розрахунок коефіцієнта варіації (CV) для інтервальної шкали (Дата розтину / Температура °C)",
        "Інтервальна шкала не має абсолютного нуля, тому відношення STD/Mean змінюється при зміні точки відліку.",
        "Використовувати міжквартильний розмах (IQR) або абсолютне стандартне відхилення."
    ),
    (
        "Розділ 4. Кореляційний аналіз",
        "Застосування кореляції Пірсона для порядкових змінних (Перебіг × Ступінь КТ)",
        "Пірсон оцінює лінійний зв’язок метричних шкал і чутливий до відстаней, відсутніх у порядковій шкалі.",
        "Розраховувати рангову кореляцію Спірмена або Тау Кендалла."
    ),
    (
        "Розділ 5. Кореляційний аналіз",
        "Обчислення кореляції Пірсона між номінальною та кількісною змінними (Стать × Вік)",
        "Числові коди номінальної шкали довільні, тому коефіцієнт Пірсона змінюється при зміні нумерації даних.",
        "Застосовувати точковий бісеріальний коефіцієнт або U-критерій Манна-Уїтні."
    ),
    (
        "Розділ 6. Аналіз текстових полів",
        "Обчислення середньої довжини символів у текстовій примітці як клінічного показника",
        "Довжина тексту відображає лише стиль ведення записів лікарем, а не медичний стан пацієнта.",
        "Вилучити текстові нотатки з кількісного аналізу або застосувати NLP-кодування."
    ),
    (
        "Розділ 7. Масштабування даних",
        "MinMax-нормалізація сталого стовпця з однаковими значеннями для всіх пацієнтів",
        "Різниця max-min дорівнює нулю, що призводить до ділення на нуль та утворення NaN без видачі помилки.",
        "Перевіряти nunique() > 1 перед нормалізацією та вилучати константні стовпці."
    ),
    (
        "Розділ 8. Кодування змінних",
        "Застосування One-Hot Encoding до порядкової шкали (Ступінь КТ / Перебіг)",
        "Створення окремих бінарних стовпців повністю втрачає інформацію про природний порядок ієрархії категорій.",
        "Застосовувати Ordinal Encoding зі збереженням природного порядку категорій."
    ),
    (
        "Розділ 9. Кодування змінних",
        "Застосування Ordinal Encoding до номінальної шкали без порядкового змісту",
        "Присвоєння послідовних чисел штучно нав'язує відношення «більше-менше» між нерівнозначними категоріями.",
        "Застосовувати One-Hot Encoding для створення бінарних ознак."
    ),
    (
        "Розділ 10. Оцінка моделей",
        "Оцінка точності прогнозу порядкової цільової змінної за допомогою метрики MSE / R²",
        "Похибка між сусідніми рангами 1 і 2 не еквівалентна похибці між 2 і 3 через неметричність шкали.",
        "Використовувати метрики класифікації (Accuracy, Confusion Matrix, F1-score)."
    )
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,Розділ 1. Описова статистика,Обчислення середнього арифметичного для порядкової шкали (Перебіг / Ступінь КТ),"Категорії порядкової шкали не мають однакових інтервалів між собою, тому середнє є арифметично безглуздим.",Застосувати медіану або моду з аналізом частотного розподілу категорій.
1,Розділ 2. Описова статистика,Обчислення середнього та стандартного відхилення для номінальної шкали (Стать / помер/жив),"Номінальні коди є лише якісними мітками, тому додавання та ділення над ними не мають математичного сенсу.","Використовувати частоти, відсоткові частки (пропорції) та моду."
2,Розділ 3. Варіативність,Розрахунок коефіцієнта варіації (CV) для інтервальної шкали (Дата розтину / Температура °C),"Інтервальна шкала не має абсолютного нуля, тому відношення STD/Mean змінюється при зміні точки відліку.",Використовувати міжквартильний розмах (IQR) або абсолютне стандартне відхилення.
3,Розділ 4. Кореляційний аналіз,Застосування кореляції Пірсона для порядкових змінних (Перебіг × Ступінь КТ),"Пірсон оцінює лінійний зв’язок метричних шкал і чутливий до відстаней, відсутніх у порядковій шкалі.",Розраховувати рангову кореляцію Спірмена або Тау Кендалла.
4,Розділ 5. Кореляційний аналіз,Обчислення кореляції Пірсона між номінальною та кількісною змінними (Стать × Вік),"Числові коди номінальної шкали довільні, тому коефіцієнт Пірсона змінюється при зміні нумерації даних.",Застосовувати точковий бісеріальний коефіцієнт або U-критерій Манна-Уїтні.
5,Розділ 6. Аналіз текстових полів,Обчислення середньої довжини символів у текстовій примітці як клінічного показника,"Довжина тексту відображає лише стиль ведення записів лікарем, а не медичний стан пацієнта.",Вилучити текстові нотатки з кількісного аналізу або застосувати NLP-кодування.
6,Розділ 7. Масштабування даних,MinMax-нормалізація сталого стовпця з однаковими значеннями для всіх пацієнтів,"Різниця max-min дорівнює нулю, що призводить до ділення на нуль та утворення NaN без видачі помилки.",Перевіряти nunique() > 1 перед нормалізацією та вилучати константні стовпці.
7,Розділ 8. Кодування змінних,Застосування One-Hot Encoding до порядкової шкали (Ступінь КТ / Перебіг),Створення окремих бінарних стовпців повністю втрачає інформацію про природний порядок ієрархії категорій.,Застосовувати Ordinal Encoding зі збереженням природного порядку категорій.
8,Розділ 9. Кодування змінних,Застосування Ordinal Encoding до номінальної шкали без порядкового змісту,Присвоєння послідовних чисел штучно нав'язує відношення «більше-менше» між нерівнозначними категоріями.,Застосовувати One-Hot Encoding для створення бінарних ознак.
9,Розділ 10. Оцінка моделей,Оцінка точності прогнозу порядкової цільової змінної за допомогою метрики MSE / R²,Похибка між сусідніми рангами 1 і 2 не еквівалентна похибці між 2 і 3 через неметричність шкали.,"Використовувати метрики класифікації (Accuracy, Confusion Matrix, F1-score)."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [34]:
import scipy.stats as stats
course_map = {'серед важ': 1, 'важкий': 2, 'вкрай важкий': 3}
df_course = df['Перебіг'].dropna().astype(str).str.strip().map(course_map)

print("1. Порядкова шкала ('Перебіг'):")
print(f"   Хибно (середнє арифметичне): {df_course.mean():.2f}")
print(f"   Правильно (медіана): {df_course.median()}, (мода): {df_course.mode()[0]}\n")

df_died = df['помер/жив'].dropna().astype(str).str.strip()

print("2. Номінальна шкала ('помер/жив'):")
print(f"   Хибно (середнє арифметичне кодів): {pd.to_numeric(df['помер/жив'], errors='coerce').mean():.2f}")
print("   Правильно (відсоткові частки категорій):")
print(df_died.value_counts(normalize=True).mul(100).round(2).to_string(), "\n")

kt_map = {'легкий': 1, 'легка': 1, 'середній': 2, 'сердній': 2, 'серед ступ': 2, 'серед ступеня': 2, 
          'важкий': 3, 'важкий ': 3, 'критичний': 4, 'вкрай важкий': 4, 'вкрай важкий ': 4}

df_pair_test = df[['Перебіг', 'ступінь КТ']].dropna().copy()
df_pair_test['Перебіг_n'] = df_pair_test['Перебіг'].astype(str).str.strip().map(course_map)
df_pair_test['КТ_n'] = df_pair_test['ступінь КТ'].astype(str).str.strip().map(kt_map)
df_pair_test = df_pair_test.dropna()

r_p, _ = stats.pearsonr(df_pair_test['Перебіг_n'], df_pair_test['КТ_n'])
r_s, _ = stats.spearmanr(df_pair_test['Перебіг_n'], df_pair_test['КТ_n'])

print("3. Порядкова × Порядкова ('Перебіг' × 'ступінь КТ'):")
print(f"   Хибно (кореляція Пірсона): {r_p:.4f}")
print(f"   Правильно (рангова кореляція Спірмена): {r_s:.4f}")

1. Порядкова шкала ('Перебіг'):
   Хибно (середнє арифметичне): 1.76
   Правильно (медіана): 2.0, (мода): 2

2. Номінальна шкала ('помер/жив'):
   Хибно (середнє арифметичне кодів): 1.50
   Правильно (відсоткові частки категорій):
помер/жив
1    50.41
2    49.59 

3. Порядкова × Порядкова ('Перебіг' × 'ступінь КТ'):
   Хибно (кореляція Пірсона): 0.6326
   Правильно (рангова кореляція Спірмена): 0.6349


---
# Етап 7. Висновки та індивідуальне питання

In [35]:
ВИСНОВКИ = [
    "Кореляція Пірсона між 'Ліжко днів' та 'Вік' виявилась дуже слабкою й статистично незначущою (r = -0.10, p = 0.28), тобто тривалість госпіталізації в цій вибірці практично не пов'язана з віком пацієнта.",
    "Стовпець 'Т вище38' насправді приховує дві різні величини під однією назвою: частина записів виміряна як «днів до госпіталізації», а частина — як «днів у стаціонарі» (наприклад, '7 днів до госп' і '2 дні в стац'), тому наївне витягання першого числа регулярним виразом змішує дві несумісні ознаки в одну статистику.",
    "З восьми стовпців мого варіанта лише три (Ліжко днів, МНВ1, Креат2) дозволяють повний арифметичний аналіз із коефіцієнтом варіації, тоді як ПІП є чистим ідентифікатором без жодної статистики, а Примітка — неструктурованим текстом, для якого має сенс лише частота."
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Кореляція Пірсона між 'Ліжко днів' та 'Вік' виявилась дуже слабкою й статистично незначущою (r = -0.10, p = 0.28), тобто тривалість госпіталізації в цій вибірці практично не пов'язана з віком пацієнта.
2. Стовпець 'Т вище38' насправді приховує дві різні величини під однією назвою: частина записів виміряна як «днів до госпіталізації», а частина — як «днів у стаціонарі» (наприклад, '7 днів до госп' і '2 дні в стац'), тому наївне витягання першого числа регулярним виразом змішує дві несумісні ознаки в одну статистику.
3. З восьми стовпців мого варіанта лише три (Ліжко днів, МНВ1, Креат2) дозволяють повний арифметичний аналіз із коефіцієнтом варіації, тоді як ПІП є чистим ідентифікатором без жодної статистики, а Примітка — неструктурованим текстом, для якого має сенс лише частота.


In [36]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()

n = len(df)
автомат_ідентифікатори = []
for c in df.columns:
    з = df[c].dropna()
    if len(з) == 0:
        continue
    completeness = len(з) / n
    unique_ratio = з.nunique() / len(з)
    if completeness >= 0.9 and unique_ratio >= 0.9:
        автомат_ідентифікатори.append((c, round(completeness, 2), round(unique_ratio, 2)))

print('Автоматично знайдені ідентифікатори:')
for c, comp, uniq in автомат_ідентифікатори:
    print(f'  {c:15s} | заповненість={comp} | унікальність={uniq}')
print('\nВсього:', len(автомат_ідентифікатори))
print()

ВІДПОВІДЬ = """У всьому файлі (77 стовпців) ідентифікаторами є рівно три стовпці: 'ПІП', 'Телефон' та 'номер і/х'. Це видно з автоматичної перевірки вище: у кожного з них заповненість близька до 100% (дані є майже для всіх 121 пацієнта), і при цьому унікальність значень теж близька до 100% (кожне значення зустрічається рівно один раз). Наївне правило "nunique дорівнює кількості непорожніх значень" саме по собі недостатнє: воно дало б хибні спрацювання на рідкісних лабораторних показниках на кшталт ІЛ6-1 чи ФЕРИТ1, які виміряні лише у 10-20 пацієнтів і тому випадково всі унікальні просто через малу вибірку, а не тому, що є ідентифікаторами. Тому правильний автоматичний критерій — поєднання двох умов одночасно: висока заповненість стовпця (майже всі рядки мають значення) ТА висока унікальність значень серед заповнених. Лише поєднання обох умов коректно відсіює рідкісні лабораторні виміри від справжніх ідентифікаторів пацієнта."""

assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Скільки стовпців файлу є ідентифікаторами? Як їх розпізнати автоматично, не дивлячись на назви?

Автоматично знайдені ідентифікатори:
  номер і/х       | заповненість=1.0 | унікальність=0.98
  ПІП             | заповненість=1.0 | унікальність=1.0
  Телефон         | заповненість=0.99 | унікальність=1.0

Всього: 3

У всьому файлі (77 стовпців) ідентифікаторами є рівно три стовпці: 'ПІП', 'Телефон' та 'номер і/х'. Це видно з автоматичної перевірки вище: у кожного з них заповненість близька до 100% (дані є майже для всіх 121 пацієнта), і при цьому унікальність значень теж близька до 100% (кожне значення зустрічається рівно один раз). Наївне правило "nunique дорівнює кількості непорожніх значень" саме по собі недостатнє: воно дало б хибні спрацювання на рідкісних лабораторних показниках на кшталт ІЛ6-1 чи ФЕРИТ1, які виміряні лише у 10-20 пацієнтів і тому випадково всі унікальні просто через малу вибірку, а не тому, що є ідентифікаторами. Тому правильний автоматичний критерій

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [39]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'синтаксис',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'код, перевірено',
    'етап 7 — висновки й відповідь':   'код, перевірено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Перевірила вручну регулярний вираз для витягання чисел на прикладі "9(5 у реан)" і "4 дні до госп". Перерахувала коефіцієнт кореляції Пірсона 0.6326 і Спірмена 0.6349 для пари Перебіг×ступінь КТ. Звірила автоматичний підрахунок ідентифікаторів (3 стовпці) ручним переглядом df.columns і df.nunique().'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [40]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 6 готова до захисту.
Соломія Лящук, ШІ-33


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.